<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# The pulse of NYC yellow-taxi pickups

A complete vector workflow on a real, public dataset. We answer:

> **Which NYC zones are busiest at which hour, and what are the dominant origin-destination flows?**

This notebook reads one month of NYC yellow-taxi trips (~3M rows, 50 MB parquet) from the public TLC mirror, joins them to taxi-zone polygons, finds each zone's peak hour, derives the top origin→destination flows as `ST_MakeLine` geometries on zone centroids, bins zone activity into Bing tiles, writes the per-zone result as **GeoParquet 1.1**, and renders the whole thing in **SedonaKepler**.

The 3M rows are the *input*. The first `groupBy` collapses them to ~5,300 `(zone, hour)` buckets, and everything downstream operates at the 263-zone scale — so the notebook fits comfortably in this single-container docker image with `DRIVER_MEM=4g` and finishes in well under two minutes once the parquet is downloaded. The same SQL would run unchanged on a year of data on a real cluster; only the `master(...)` URL changes.

**Data sources** — taxi-zone polygons ship with the docker image (NYC TLC, public). Trip data is fetched at runtime from the [TLC Cloudfront mirror](https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet).

<!-- requires-network: true -->

*This notebook requires outbound network access to download the trip parquet (~50 MB). Set `SEDONA_NOTEBOOK_OFFLINE=1` to skip it under sandboxed CI.*

## 1. Connect to Spark through SedonaContext

`SedonaContext.create(...)` registers the Sedona spatial functions on top of an existing Spark session and returns it. Everything below is plain Spark SQL with the `ST_*` and `RS_*` catalog enabled.

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

## 2. Load taxi-zone polygons and reproject to WGS84

The shipped shapefile is in **EPSG:2263** (NAD83 / NY State Plane Long Island, US-feet). For tile binning and Kepler we want lon/lat. Since 1.9, `ST_Transform` is backed by **proj4sedona** and accepts EPSG codes, full WKT2, PROJ strings, and PROJJSON interchangeably.

In [ ]:
zones_raw = sedona.read.format("shapefile").load("data/nyc_taxi_zones")
zones = zones_raw.selectExpr(
    "LocationID as location_id",
    "zone",
    "borough",
    "ST_Transform(geometry, 'epsg:2263', 'epsg:4326') as geometry",
)
zones.cache()
print(f"{zones.count()} taxi zones")
zones.show(5, truncate=40)

## 3. Pull one month of yellow-taxi trips

Spark cannot currently read Parquet directly from an `https://` URL through its Hadoop FileSystem layer, so we cache the file to `/tmp` once and point Spark at the local copy. The TLC keeps `pickup_longitude`/`pickup_latitude` out of recent releases for privacy, so trips are zoned by `PULocationID`/`DOLocationID` — those join to the polygons above by ID.

In [ ]:
import os
import urllib.request

TRIP_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
)
TRIP_PATH = "/tmp/yellow_tripdata_2024-01.parquet"

if not os.path.exists(TRIP_PATH):
    print(f"downloading {TRIP_URL} ...")
    urllib.request.urlretrieve(TRIP_URL, TRIP_PATH)

trips = sedona.read.parquet(TRIP_PATH).selectExpr(
    "PULocationID as pu_zone",
    "DOLocationID as do_zone",
    "hour(tpep_pickup_datetime) as pickup_hour",
    "tpep_pickup_datetime as pickup_ts",
)
trips = trips.where("pickup_ts >= '2024-01-01' AND pickup_ts < '2024-02-01'")
print(f"{trips.count():,} trips")
trips.show(5)

## 4. Find each zone's peak hour

Aggregate trips per `(zone, hour)` and use `MAX_BY` to pull the busiest hour per zone in a single pass. We label each zone with a coarse bucket (morning / midday / evening / nightlife) so the map at the end is read-at-a-glance.

In [ ]:
trips.createOrReplaceTempView("trips")

zone_hour = sedona.sql("""
    SELECT pu_zone, pickup_hour, COUNT(*) AS trips
    FROM trips
    GROUP BY pu_zone, pickup_hour
""")
zone_hour.createOrReplaceTempView("zone_hour")

zone_profile = sedona.sql("""
    SELECT pu_zone,
           SUM(trips)                  AS total_trips,
           MAX_BY(pickup_hour, trips)  AS peak_hour,
           CASE
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 5  AND 10 THEN 'morning'
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 11 AND 15 THEN 'midday'
               WHEN MAX_BY(pickup_hour, trips) BETWEEN 16 AND 19 THEN 'evening'
               ELSE                                                   'nightlife'
           END                         AS peak_bucket
    FROM zone_hour
    GROUP BY pu_zone
""")
zone_profile.show(10)

## 5. Bin each zone's centroid into a Bing-tile quadkey

`ST_BingTileAt(longitude, latitude, zoom)` is new in 1.9 and produces a quadkey string for the Bing tile that contains a point. Combined with `ST_Centroid`, it gives a free "which neighborhood-sized tile is this zone in?" feature without bringing in an external H3/S2 library.

In [ ]:
zones.createOrReplaceTempView("zones")
zone_profile.createOrReplaceTempView("zone_profile")

zones_with_pulse = sedona.sql("""
    SELECT z.location_id, z.zone, z.borough,
           p.total_trips, p.peak_hour, p.peak_bucket,
           z.geometry,
           ST_Centroid(z.geometry)                                  AS centroid,
           ST_BingTileAt(ST_X(ST_Centroid(z.geometry)),
                         ST_Y(ST_Centroid(z.geometry)), 14)         AS tile_quadkey
    FROM zones z
    JOIN zone_profile p ON z.location_id = p.pu_zone
""")
zones_with_pulse.cache()
zones_with_pulse.select(
    "zone", "borough", "total_trips", "peak_hour", "peak_bucket", "tile_quadkey"
).show(10, truncate=30)

## 6. Build top origin-destination flow lines

Take the 30 most-travelled `(origin, destination)` zone pairs and draw a `LineString` from origin centroid to destination centroid. `ST_MakeLine(ST_Centroid(...), ST_Centroid(...))` is the idiomatic way to derive flow geometries from a zone-attributed OD matrix.

In [ ]:
od = sedona.sql("""
    SELECT pu_zone, do_zone, COUNT(*) AS trips
    FROM trips
    WHERE pu_zone <> do_zone
    GROUP BY pu_zone, do_zone
    ORDER BY trips DESC
    LIMIT 30
""")
od.createOrReplaceTempView("od")

flows = sedona.sql("""
    SELECT od.pu_zone, od.do_zone, od.trips,
           pu.zone AS pu_name, do_.zone AS do_name,
           ST_MakeLine(ST_Centroid(pu.geometry),
                       ST_Centroid(do_.geometry)) AS flow_geom
    FROM od
    JOIN zones pu  ON pu.location_id  = od.pu_zone
    JOIN zones do_ ON do_.location_id = od.do_zone
""")
flows.select("pu_name", "do_name", "trips").show(10, truncate=30)

## 7. Persist per-zone aggregates as GeoParquet 1.1

Sedona auto-populates **covering-bbox metadata** and **projjson CRS** from the geometry SRID; downstream readers can filter on a bounding box without scanning every row. We round-trip through a read-back to confirm the schema lands.

In [ ]:
import shutil

out_path = "/tmp/nyc_taxi_zone_pulse.parquet"
if os.path.exists(out_path):
    shutil.rmtree(out_path)

(
    zones_with_pulse.select(
        "location_id",
        "zone",
        "borough",
        "total_trips",
        "peak_hour",
        "peak_bucket",
        "tile_quadkey",
        "geometry",
    )
    .coalesce(1)
    .write.format("geoparquet")
    .save(out_path)
)

sedona.read.format("geoparquet").load(out_path).select(
    "zone", "borough", "total_trips", "peak_bucket"
).show(5, truncate=30)

## 8. Render three layers on a single SedonaKepler map

- **`zones_by_peak`** — zone polygons coloured by peak-hour bucket
- **`zone_centroids`** — points scaled by trip count
- **`top_od_flows`** — the busiest 30 origin-destination flowlines

All three layers share the same `geometry` column convention; `SedonaKepler.add_df` handles the Spark-to-Kepler conversion under the hood.

In [ ]:
from sedona.spark import SedonaKepler

zones_layer = sedona.read.format("geoparquet").load(out_path).drop("tile_quadkey")
centroids_layer = zones_with_pulse.selectExpr(
    "zone", "borough", "total_trips", "peak_bucket", "centroid as geometry"
)
flows_layer = flows.selectExpr("pu_name", "do_name", "trips", "flow_geom as geometry")

kepler_map = SedonaKepler.create_map(df=zones_layer, name="zones_by_peak")
SedonaKepler.add_df(kepler_map, centroids_layer, name="zone_centroids")
SedonaKepler.add_df(kepler_map, flows_layer, name="top_od_flows")
kepler_map

## What just happened?

We turned 3 million raw trip records into a labelled spatial dataset in eight steps:

1. Reprojected zone polygons from NY State Plane to WGS84 with `ST_Transform` (proj4sedona).
2. Pulled a month of TLC trips from the public Cloudfront mirror.
3. Aggregated by `(zone, hour)` and bucketed each zone's peak hour with a single `MAX_BY` window.
4. Bing-tiled zone centroids via the new `ST_BingTileAt`.
5. Built the top-30 origin-destination flow geometries with `ST_MakeLine` over `ST_Centroid` outputs.
6. Wrote the per-zone result as **GeoParquet 1.1** with auto covering-bbox metadata, then read it back.
7. Visualized zones, centroids, and flowlines as three layers on a single SedonaKepler map.

Sedona ran the same SQL on a single laptop here that it would run across a thousand-node cluster on a year of TLC data — no API change, just more workers.